# Simulate a 2 neuron E-I system

... to demonstrate the concept of an effective weight in the context of multiple Pathways and the compatibility of SAL with Dale's law

In [ ]:
from pathlib import Path
import pickle
import numpy as np
from scipy.special import expit
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from mpl_toolkits.axes_grid1 import make_axes_locatable
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm
import psutil

from neuralsampling.network import rect_kernel
from neuralsampling.eisystem import EISystem

In [ ]:
# define the style etc.
mpl.style.use("../../mystyle.mpl")

In [ ]:
FIG_DIR = Path("figs/")
FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data/ei_system")
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
SIMULATE = True

## Define functions

In [ ]:
def plot_ppd(fig, ax, range_weights, dw_12, dw_21, color, w_eff_12=None, w_eff_21=None):
    """Plotting routine for phase plane diagrams
    Takes the Delta weights (2-dim array) and the weight range (1-dim array)
    and plots the arrow field and the update magnitude as color map. Marks also
    all attractors and repellors.
    """
    ax.grid()
    abs_stdp = np.sqrt(dw_12**2 + dw_21**2)

    ax.set_aspect("equal")
    cm = plt.get_cmap("GnBu")

    range_weights_eff_12 = w_eff_12[1](range_weights)
    range_weights_eff_21 = w_eff_21[1](range_weights)

    im = ax.contourf(
        range_weights_eff_12,
        range_weights_eff_21,
        abs_stdp / np.max(abs_stdp),
        15,
        alpha=0.9,
        cmap=cm,
    )

    quiv = ax.quiver(
        range_weights_eff_12[::2],
        range_weights_eff_21[::2],
        dw_12[::2, ::2],
        dw_21[::2, ::2],
        pivot="mid",
    )

    # ax.contour(range_weights, range_weights, dw_12, [0.], colors='red')
    ax.contour(range_weights_eff_12, range_weights_eff_21, dw_21, [0.0], colors=color)

    ax.set_xlabel(r"$W_{ij}^\mathrm{eff}$")
    ax.set_ylabel(r"$W_{ji}^\mathrm{eff}$", labelpad=-5)
    ax.set_xlim(range_weights_eff_12[0], range_weights_eff_12[-1])
    ax.set_ylim(range_weights_eff_21[0], range_weights_eff_21[-1])

    # print(ax.get_xlim())

    # ax.plot([range_weights[0], range_weights[-1]],
    #         [range_weights[0], range_weights[-1]], 'k--')

    if w_eff_12 is not None:
        ax_eff_top = ax.secondary_xaxis("top", functions=w_eff_12)
        ax_eff_top.set_xlabel(r"$W_{ji}^\mathrm{EE}$")

    if w_eff_21 is not None:
        ax_eff_right = ax.secondary_yaxis("right", functions=w_eff_21)
        ax_eff_right.set_ylabel(r"$W_{ij}^\mathrm{EE}$")

    ax.xaxis.set_major_locator(ticker.MultipleLocator(0.5))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(0.5))
    ax_eff_top.xaxis.set_major_locator(ticker.MultipleLocator(0.5))
    ax_eff_right.yaxis.set_major_locator(ticker.MultipleLocator(0.5))

    divider = make_axes_locatable(ax)
    cax = divider.append_axes("bottom", size="5%", pad=0.5)

    fig.colorbar(
        im,
        label=r"$\sqrt{\Delta W_{ij}^2 + \Delta W_{ji}^2}$",
        cax=cax,
        ticks=[0.0, 0.5, 1.0],
        # pad=0.25,
        orientation="horizontal",
    )

    # ax.set_xticks(ax.get_yticks())

    return fig, ax


def change_axis_color(ax, color, spine="bottom"):
    ax.spines[spine].set_color(color)
    ax.tick_params(axis="x", which="both", colors=color)
    ax.xaxis.label.set_color(color)
    for label in ax.get_xticklabels(minor=False):
        label.set_color(color)

# make a PPD with symmetric parameterization

Proof of principle, not used in the paper!

In [ ]:
Ws = np.arange(0, 2.1, 0.1)
n = len(Ws)
stdp_12 = np.zeros((n, n))
stdp_21 = np.zeros((n, n))
r_1 = np.zeros((n, n))
r_2 = np.zeros((n, n))


def worker(args):
    i, j = args
    w_12 = Ws[i]
    w_21 = Ws[j]
    weights = {
        "e2_e1": w_12,
        "e1_e2": w_21,
        "i1_e1": 4.0,
        "i2_e2": 4.0,
        "e2_i1": -1.0,
        "e1_i2": -1.0,
    }
    bias = {"e1": 0.0, "e2": 0.0, "i1": -2.0, "i2": -2.0}
    eisys = EISystem(weights, bias, tref=25, tsyn=25, kernel=rect_kernel)
    eisys.simulate(2_000_000)
    eisys.calc_stdp()
    stdp_21, stdp_12 = eisys.stdp_e2_e1, eisys.stdp_e1_e2
    del eisys
    return (i, j, stdp_12, stdp_21)


index_pairs = [(i, j) for i in range(n) for j in range(n)]

if SIMULATE:
    with ProcessPoolExecutor(max_workers=psutil.cpu_count(logical=False)) as executor:
        results = list(executor.map(worker, index_pairs))
    for i, j, val_12, val_21 in results:
        stdp_12[i, j] = val_12
        stdp_21[i, j] = val_21
    with open(DATA_DIR / "ei_system_data.pkl", "wb") as f:
        pickle.dump((stdp_12, stdp_21), f)
else:
    with open(DATA_DIR / "ei_system_data.pkl", "rb") as f:
        stdp_12, stdp_21 = pickle.load(f)

# for the transformation into effective weights:
weights = {
    "e2_e1": 0.0,
    "e1_e2": 0.0,
    "i1_e1": 4.0,
    "i2_e2": 4.0,
    "e2_i1": -1.0,
    "e1_i2": -1.0,
}
bias = {"e1": 0.0, "e2": 0.0, "i1": -2.0, "i2": -2.0}
eisys = EISystem(weights, bias, tref=25, tsyn=25, kernel=rect_kernel)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8 / 2.54, 6 / 2.54))

plot_ppd(
    fig,
    ax,
    Ws,
    stdp_12,
    stdp_21,
    "red",
    w_eff_12=(
        eisys.transform_effective_weights_12_inverse,
        eisys.transform_effective_weights_12,
    ),
    w_eff_21=(
        eisys.transform_effective_weights_21_inverse,
        eisys.transform_effective_weights_21,
    ),
)

# plt.tight_layout()

In [ ]:
fig.savefig(FIG_DIR / "ei_system_ppd.png", bbox_inches="tight")
fig.savefig(FIG_DIR / "ei_system_ppd.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / "ei_system_ppd.svg", bbox_inches="tight")

# now with a bit of noise (i.e. not perfectly symmetric weights in both pathways):

This is the one that is used in the paper!

In [ ]:
Ws = np.arange(0, 2.1, 0.1)
n = len(Ws)
stdp_12 = np.zeros((n, n))
stdp_21 = np.zeros((n, n))
r_1 = np.zeros((n, n))
r_2 = np.zeros((n, n))
T_SIM = 2_000_000

rng = np.random.default_rng(97)

weights = {
    "e2_e1": 0.0,
    "e1_e2": 0.0,
    "i1_e1": rng.normal(4.0, 0.2),
    "i2_e2": rng.normal(4.0, 0.2),
    "e2_i1": rng.normal(-1.0, 0.1),
    "e1_i2": rng.normal(-1.0, 0.1),
}
bias = {
    "e1": 0.0,
    "e2": 0.0,
    "i1": rng.normal(-2.0, 0.2),
    "i2": rng.normal(-2.0, 0.2),
}


def worker(args):
    i, j = args
    w_12 = Ws[i]
    w_21 = Ws[j]
    weights["e2_e1"], weights["e1_e2"] = w_12, w_21
    eisys = EISystem(weights, bias, tref=25, tsyn=25, kernel=rect_kernel)
    eisys.simulate(T_SIM)
    eisys.calc_stdp()
    stdp_12, stdp_21 = eisys.stdp_e2_e1, eisys.stdp_e1_e2
    del eisys
    return (i, j, stdp_12, stdp_21)


index_pairs = [(i, j) for i in range(n) for j in range(n)]

if SIMULATE:
    with ProcessPoolExecutor(max_workers=psutil.cpu_count(logical=False)) as executor:
        results = list(executor.map(worker, index_pairs))
    for i, j, val_12, val_21 in results:
        stdp_12[i, j] = val_12
        stdp_21[i, j] = val_21
    with open(DATA_DIR / "ei_system_data_noise.pkl", "wb") as f:
        pickle.dump((stdp_12, stdp_21), f)
else:
    with open(DATA_DIR / "ei_system_data_noise.pkl", "rb") as f:
        stdp_12, stdp_21 = pickle.load(f)


# for the transformation into effective weights:
eisys = EISystem(weights, bias, tref=25, tsyn=25, kernel=rect_kernel)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5.0 / 2.54, 6.5 / 2.54))

plot_ppd(
    fig,
    ax,
    Ws,
    stdp_21,
    stdp_12,
    "red",
    w_eff_12=(
        eisys.transform_effective_weights_12_inverse,
        eisys.transform_effective_weights_12,
    ),
    w_eff_21=(
        eisys.transform_effective_weights_21_inverse,
        eisys.transform_effective_weights_21,
    ),
)

In [ ]:
fig.savefig(FIG_DIR / "ei_system_ppd_noised.png", bbox_inches="tight")
fig.savefig(FIG_DIR / "ei_system_ppd_noised.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / "ei_system_ppd_noised.svg", bbox_inches="tight")

In [ ]:
w_21 = np.arange(0.0, 2.1, 0.25)
T_SIM = 500_000
rng = np.random.default_rng(97)

weights = {
    "e2_e1": 0.0,
    "e1_e2": 0.0,
    "i1_e1": rng.normal(4.0, 0.2),
    "i2_e2": rng.normal(4.0, 0.2),
    "e2_i1": rng.normal(-1.0, 0.1),
    "e1_i2": rng.normal(-1.0, 0.1),
}
bias = {
    "e1": 0.0,
    "e2": 0.0,
    "i1": rng.normal(-2.0, 0.2),
    "i2": rng.normal(-2.0, 0.2),
}

# direct system:
direct_system = EISystem(weights, bias, tref=25, tsyn=25, kernel=rect_kernel)
direct_system.w_e1_e2 = 0.0
direct_system.w_e1_i2 = 0.0
direct_system.w_e2_i1 = 0.0
dales_system = EISystem(weights, bias, tref=25, tsyn=25, kernel=rect_kernel)
dales_system.w_e1_e2 = 0.0
dales_system.w_e1_i2 = 0.0

rates_dale = []
rates_direct = []
for w in w_21:
    dales_system.w_e2_e1 = w
    dales_system.simulate(T_SIM)
    dales_system.calc_rates()
    rates_dale.append(dales_system.rate_e2)

    w_eff = dales_system.transform_effective_weights_21(w)
    direct_system.w_e2_e1 = w_eff
    direct_system.simulate(500_000)
    direct_system.calc_rates()
    rates_direct.append(direct_system.rate_e2)

In [ ]:
fig, ax = plt.subplots(figsize=(5 / 2.54, 3.0 / 2.54))

ax.plot(
    w_21,
    rates_dale,
    linestyle="",
    marker="o",
    markerfacecolor="none",
    markeredgewidth=2,
    label="E-I",
)
ax.plot(w_21, rates_direct, linestyle="", marker="s", zorder=1, label="direct")

ax_upper = ax.secondary_xaxis(
    "top",
    functions=(
        dales_system.transform_effective_weights_21,
        dales_system.transform_effective_weights_21_inverse,
    ),
)

ax.set_xlim(-0.2, 2.2)
ax.set_ylim(0.36, 0.64)
ax.set_ylabel(r"$r^E_j\ [\tau_\mathrm{ref}^{-1}]$")
ax.set_xlabel(r"$W_{ij}^\mathrm{EE}$")
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax_upper.set_xlabel(r"$W_{ji}^\mathrm{eff}$")
ax_upper.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.spines["top"].set_visible(False)
change_axis_color(ax, "C0")
change_axis_color(ax_upper, "C1", spine="top")

ax.legend()

In [ ]:
fig.savefig(FIG_DIR / "ei_system_eff_weight.png", bbox_inches="tight")
fig.savefig(FIG_DIR / "ei_system_eff_weight.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / "ei_system_eff_weight.svg", bbox_inches="tight")